# CAD parameter validation

Small demo: load **Cameo** requirements and **CATIA** parameters, compare bounded requirements to CAD values, show an HTML report.

You will:
1. Load bundled JSON from disk **or** read extraction artifacts from the Istari Digital Platform
2. Generate HTML and JUnit XML validation reports
3. Upload the JUnit report as an artifact on the platform
4. Optionally push an updated parameter back to the CATIA model on the platform

Prerequisites: `istari_labs_helpers` kernel, `samples/.env` — see [Chaining jobs](../chaining_jobs.ipynb).

## 0 · Load from disk

> **Run this section** when working from bundled files in this folder. **Skip it** when you will read extraction outputs from the Istari Digital Platform in the sections below.

In [ ]:
# Run this cell if you work from files; skip it to read from the Istari Digital Platform.

from pathlib import Path

from validation_lib import load_json

USE_LOCAL_FILES = True
NOTEBOOK_DIR = Path.cwd()

requirements = load_json(NOTEBOOK_DIR / "requirements-uav-length2.json")
parameters = load_json(NOTEBOOK_DIR / "catia-parameters-length1800.json")
print(
    f"Loaded local files — {len(requirements)} requirement elements, "
    f"{len(parameters)} CATIA parameters"
)

## 1 · Connect

In [ ]:
from pathlib import Path

from IPython.display import HTML, display
from istari_labs_helpers import IstariPlatform, JobDefinition

from validation_lib import fetch_artifact, preview_junit_html, render_html, render_junit_xml, run

NOTEBOOK_DIR = Path.cwd()
USE_LOCAL_FILES = globals().get("USE_LOCAL_FILES", False)

if USE_LOCAL_FILES:
    print("Skipping platform connect — using local files from step 0.")
else:
    platform = IstariPlatform.from_env(str(NOTEBOOK_DIR.parent / ".env"))
    current_user = platform.client.get_current_user()
    print(platform)
    print(f"User ID: {current_user.id}")

## 2 · Job IDs

Paste completed extraction **job IDs** from the Istari Digital web app (Jobs tab).

In [ ]:
USE_LOCAL_FILES = globals().get("USE_LOCAL_FILES", False)

if USE_LOCAL_FILES:
    print("Skipping job IDs — using local files from step 0.")
else:
    CAMEO_EXTRACTION_JOB_ID = "7d1d08bd-c1ec-4cc2-8ae4-39c5d058699a"
    CATIA_EXTRACTION_JOB_ID = "81e39581-5a47-44e0-ad91-b7c60e2d5cc6"

    REQUIREMENTS_ARTIFACT = "requirements.json"
    PARAMETERS_ARTIFACT = "parameters.json"

## 3 · Retrieve extraction outputs

- **Requirements** — Cameo extraction (`requirements.json`)
- **Parameters** — CATIA extraction (`catia_parameters.json`)

Each JSON file is read from the job's pinned product revision via `fetch_artifact()`.

In [ ]:
USE_LOCAL_FILES = globals().get("USE_LOCAL_FILES", False)

if USE_LOCAL_FILES:
    print("Skipping platform retrieval — using local files from step 0.")
else:
    requirements = fetch_artifact(platform, CAMEO_EXTRACTION_JOB_ID, REQUIREMENTS_ARTIFACT)
    parameters = fetch_artifact(platform, CATIA_EXTRACTION_JOB_ID, PARAMETERS_ARTIFACT)
    print(f"Loaded {REQUIREMENTS_ARTIFACT} and {PARAMETERS_ARTIFACT}")

## 4 · Generate HTML validation report

In [ ]:
rows = run(requirements, parameters)
html = render_html(rows)

(NOTEBOOK_DIR / "validation_report.html").write_text(html, encoding="utf-8")
print(f"{len(rows)} comparable checks — all passed" if rows and all(r.passed for r in rows) else f"{len(rows)} checks")
display(HTML(html))

## 5 · Generate JUnit XML report

Write a JUnit XML file (the format Istari Digital previews in the web app when you upload it as an artifact). The HTML preview below mirrors that viewer inside the notebook.

In [ ]:
JUNIT_REPORT_NAME = "validation_results.xml"

junit_path = NOTEBOOK_DIR / JUNIT_REPORT_NAME
junit_xml = render_junit_xml(rows)
junit_path.write_text(junit_xml, encoding="utf-8")

n_fail = sum(not r.passed for r in rows)
print(f"Wrote {JUNIT_REPORT_NAME} — {len(rows)} tests, {n_fail} failures")
display(HTML(preview_junit_html(rows, xml_path=junit_path)))

## 6 · Upload JUnit report

Register the report on the Istari Digital Platform as an artifact. Open it in the **Files** page to use the built-in JUnit preview.

In [ ]:
USE_LOCAL_FILES = globals().get("USE_LOCAL_FILES", False)

if USE_LOCAL_FILES:
    print("Skipping upload — using local files from step 0.")
else:
    from istari_digital_client import Configuration, V3Client
    from istari_digital_client.v3.models.resource_type_dto import ResourceTypeDto

    config = platform.client.api_client.configuration
    v3 = V3Client(config)

    report = v3.create_resource(
        path=junit_path,
        resource_type=ResourceTypeDto.ARTIFACT,
        display_name=JUNIT_REPORT_NAME,
        description="CAD parameter validation — JUnit summary",
    )
    print(f"Uploaded {JUNIT_REPORT_NAME!r} → resource_id={report.resource_id}")

## 7 · Update wing length on CATIA model

Run `@istari:update_parameters` on the CATIA assembly. Set `CATIA_MODEL_ID` to your model UUID from the platform (Models tab). Requires a **dassault_catia_v5** agent on Windows.

In [ ]:
USE_LOCAL_FILES = globals().get("USE_LOCAL_FILES", False)

if USE_LOCAL_FILES:
    print("Skipping CATIA update — using local files from step 0.")
else:
    CATIA_MODEL_ID = "paste-your-catia-model-uuid-here"

    WING_LENGTH_PARAM = r"SA ISTARI_ONE\WING.2\WING_LENGTH"
    WING_LENGTH_MM = 1600  # mm — choose a value within your requirement bounds

In [ ]:
USE_LOCAL_FILES = globals().get("USE_LOCAL_FILES", False)

if USE_LOCAL_FILES:
    print("Skipping CATIA update — using local files from step 0.")
else:
    catia_model = platform.get_model(CATIA_MODEL_ID)

    update_def = JobDefinition(
        function="@istari:update_parameters",
        tool_name="dassault_catia_v5",
        tool_version="6R2023",
        operating_system="Windows 10",
        input_json_data={
            "parameters": {
                WING_LENGTH_PARAM: WING_LENGTH_MM,
            },
        },
    )

    update_job = catia_model.submit_job(update_def)
    print(f"Submitted update job {update_job.id}; polling...")

    update_job.wait(
        timeout=900,
        on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
    ).on_success()

    print(f"WING_LENGTH set to {WING_LENGTH_MM} mm — job {update_job.id} completed")